In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuración de catálogo y esquemas
CATALOG = "smart_claims"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print("✓ Configuración lista")
print(f"Catálogo: {CATALOG}")
print(f"Esquema origen: {SILVER_SCHEMA}")
print(f"Esquema destino: {GOLD_SCHEMA}")
print("\n🎯 Objetivo: Crear tablas integradas para análisis de negocio")

✓ Configuración lista
Catálogo: smart_claims
Esquema origen: silver
Esquema destino: gold

🎯 Objetivo: Crear tablas integradas para análisis de negocio


In [0]:
# =====================================================
# PASO A: gold.aggregated_telematics
# =====================================================
# Objetivo: Agrupar telemática por vehículo (chassis_no)
#           y obtener indicadores promedio
# =====================================================

print("="*70)
print("🚗 PASO A: Agregación de Telemática por Vehículo")
print("="*70)

# Leer tabla Silver
df_telematics = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.telematics_clean")

print(f"\n📊 Registros en telematics_clean: {df_telematics.count():,}")
print(f"   Vehículos únicos: {df_telematics.select('chassis_no').distinct().count():,}")

# Agregaciones por vehículo
df_aggregated_telematics = df_telematics \
    .groupBy("chassis_no") \
    .agg(
        # Velocidad
        avg("speed").alias("avg_speed"),
        min("speed").alias("min_speed"),
        max("speed").alias("max_speed"),
        stddev("speed").alias("std_speed"),
        
        # Coordenadas promedio (centro de actividad del vehículo)
        avg("latitude").alias("avg_latitude"),
        avg("longitude").alias("avg_longitude"),
        
        # Rango de coordenadas (dispersión geográfica)
        (max("latitude") - min("latitude")).alias("latitude_range"),
        (max("longitude") - min("longitude")).alias("longitude_range"),
        
        # Timestamps
        min("event_timestamp").alias("first_event_timestamp"),
        max("event_timestamp").alias("last_event_timestamp"),
        
        # Conteo de eventos
        count("*").alias("total_events")
    ) \
    .withColumn(
        # Calcular días de actividad del vehículo
        "days_active",
        datediff(col("last_event_timestamp"), col("first_event_timestamp"))
    ) \
    .withColumn(
        # Eventos promedio por día
        "avg_events_per_day",
        when(col("days_active") > 0, col("total_events") / col("days_active")).otherwise(col("total_events"))
    )

# Redondear valores decimales para legibilidad
df_aggregated_telematics = df_aggregated_telematics \
    .withColumn("avg_speed", round(col("avg_speed"), 2)) \
    .withColumn("min_speed", round(col("min_speed"), 2)) \
    .withColumn("max_speed", round(col("max_speed"), 2)) \
    .withColumn("std_speed", round(col("std_speed"), 2)) \
    .withColumn("avg_latitude", round(col("avg_latitude"), 6)) \
    .withColumn("avg_longitude", round(col("avg_longitude"), 6)) \
    .withColumn("latitude_range", round(col("latitude_range"), 6)) \
    .withColumn("longitude_range", round(col("longitude_range"), 6)) \
    .withColumn("avg_events_per_day", round(col("avg_events_per_day"), 2))

# Guardar tabla Gold
df_aggregated_telematics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.aggregated_telematics")

print(f"\n✓ Tabla {CATALOG}.{GOLD_SCHEMA}.aggregated_telematics creada")
print(f"  Vehículos agregados: {df_aggregated_telematics.count():,}")
print("\n📊 Indicadores calculados por vehículo:")
print("   • Velocidad: promedio, mínima, máxima, desviación estándar")
print("   • Coordenadas: centro de actividad (lat/lon promedio)")
print("   • Dispersión geográfica: rango de latitud/longitud")
print("   • Actividad temporal: días activos, eventos totales, eventos/día")

# Mostrar muestra
print("\n🔍 Muestra de datos agregados:")
display(df_aggregated_telematics.limit(5))

🚗 PASO A: Agregación de Telemática por Vehículo

📊 Registros en telematics_clean: 780,060
   Vehículos únicos: 11,585

✓ Tabla smart_claims.gold.aggregated_telematics creada
  Vehículos agregados: 11,585

📊 Indicadores calculados por vehículo:
   • Velocidad: promedio, mínima, máxima, desviación estándar
   • Coordenadas: centro de actividad (lat/lon promedio)
   • Dispersión geográfica: rango de latitud/longitud
   • Actividad temporal: días activos, eventos totales, eventos/día

🔍 Muestra de datos agregados:


chassis_no,avg_speed,min_speed,max_speed,std_speed,avg_latitude,avg_longitude,latitude_range,longitude_range,first_event_timestamp,last_event_timestamp,total_events,days_active,avg_events_per_day
YV2RS02D7EA767675,32.15,0.49,57.18,18.12,40.804515,-73.925425,0.039025,0.052054,2015-12-04T10:50:00.000Z,2015-12-04T11:49:00.000Z,60,0,60.0
JHMFD16777S410994,32.15,0.49,57.18,18.12,40.804515,-73.925425,0.039025,0.052054,2015-12-04T10:50:00.000Z,2015-12-04T11:49:00.000Z,60,0,60.0
JHMAP11307S200053,32.15,0.49,57.18,18.12,40.804515,-73.925425,0.039025,0.052054,2015-12-04T10:50:00.000Z,2015-12-04T11:49:00.000Z,60,0,60.0
KNDMB233676183019,32.15,0.49,57.18,18.12,40.804515,-73.925425,0.039025,0.052054,2015-12-04T10:50:00.000Z,2015-12-04T11:49:00.000Z,60,0,60.0
JTDBW923794025112...,35.36,0.11,71.72,20.01,40.797944,-73.936665,0.190977,0.201836,2015-03-05T16:40:00.000Z,2015-12-04T11:49:00.000Z,240,274,0.88


In [0]:
# =====================================================
# PASO B: gold.customer_claim_policy
# =====================================================
# Objetivo: Consolidar información de reclamo + póliza + cliente
# =====================================================

print("="*70)
print("📄 PASO B: Integración de Reclamos, Pólizas y Clientes")
print("="*70)

# Leer tablas Silver
df_claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.claims_clean")
df_policies = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.policies_clean")
df_customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers_clean")

print(f"\n📊 Registros en cada tabla:")
print(f"   claims_clean: {df_claims.count():,}")
print(f"   policies_clean: {df_policies.count():,}")
print(f"   customers_clean: {df_customers.count():,}")

# Paso 1: Unir claims con policies (por policy_no)
print("\n🔗 Paso 1: Unir reclamos con pólizas...")
df_claim_policy = df_claims.join(
    df_policies,
    df_claims.policy_no == df_policies.policy_no,
    "left"  # Left join para conservar todos los reclamos
)

# Seleccionar columnas relevantes (evitar duplicados)
df_claim_policy = df_claim_policy.select(
    # Información del reclamo
    df_claims.claim_no,
    df_claims.policy_no,
    df_claims.claim_date,
    df_claims.incident_date,
    df_claims.incident_type,
    df_claims.incident_severity,
    df_claims.collision_type,
    df_claims.hour.alias("incident_hour"),
    df_claims.injury,
    df_claims.property,
    df_claims.vehicle,
    df_claims.total.alias("claim_amount"),
    df_claims.age.alias("claimant_age"),
    df_claims.insured_relationship,
    df_claims.license_issue_date,
    df_claims.number_of_vehicles_involved,
    df_claims.number_of_witnesses,
    df_claims.months_as_customer,
    df_claims.fraud_reported,
    df_claims.valid_total,
    df_claims.valid_claim_date,
    
    # Información de la póliza
    df_policies.customer_id,
    df_policies.policy_type,
    df_policies.policy_issue_date,
    df_policies.policy_effective_date,
    df_policies.policy_expiry_date,
    df_policies.make.alias("vehicle_make"),
    df_policies.model.alias("vehicle_model"),
    df_policies.model_year.alias("vehicle_year"),
    df_policies.chassis_no,
    df_policies.use_of_vehicle,
    df_policies.product.alias("policy_product"),
    df_policies.sum_insured,
    df_policies.premium,
    df_policies.deductible,
    df_policies.valid_dates.alias("policy_valid_dates"),
    df_policies.valid_premium
)

print(f"   ✓ Reclamos unidos con pólizas: {df_claim_policy.count():,} registros")

# Paso 2: Unir con customers (por customer_id)
print("\n🔗 Paso 2: Unir con información de clientes...")
df_customer_claim_policy = df_claim_policy.join(
    df_customers,
    df_claim_policy.customer_id == df_customers.customer_id,
    "left"  # Left join para conservar todos los reclamos
)

# Seleccionar todas las columnas anteriores + información del cliente
df_customer_claim_policy = df_customer_claim_policy.select(
    # Mantener todas las columnas anteriores
    df_claim_policy["*"],
    
    # Agregar información del cliente
    df_customers.first_name,
    df_customers.last_name,
    df_customers.date_of_birth,
    df_customers.borough,
    df_customers.neighborhood,
    df_customers.zip_code,
    df_customers.address
)

# Guardar tabla Gold
df_customer_claim_policy.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy")

print(f"\n✓ Tabla {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy creada")
print(f"  Registros totales: {df_customer_claim_policy.count():,}")
print(f"  Columnas: {len(df_customer_claim_policy.columns)}")
print("\n📊 Información consolidada:")
print("   • Datos del reclamo (fecha, tipo, severidad, montos)")
print("   • Datos de la póliza (vehículo, cobertura, premium)")
print("   • Datos del cliente (nombre, ubicación, demografía)")

# Mostrar muestra
print("\n🔍 Muestra de datos integrados:")
display(df_customer_claim_policy.select(
    "claim_no", "first_name", "last_name", "vehicle_make", "vehicle_model",
    "incident_type", "claim_amount", "premium"
).limit(5))

📄 PASO B: Integración de Reclamos, Pólizas y Clientes

📊 Registros en cada tabla:
   claims_clean: 12,991
   policies_clean: 12,237
   customers_clean: 7,061

🔗 Paso 1: Unir reclamos con pólizas...
   ✓ Reclamos unidos con pólizas: 12,992 registros

🔗 Paso 2: Unir con información de clientes...

✓ Tabla smart_claims.gold.customer_claim_policy creada
  Registros totales: 12,992
  Columnas: 44

📊 Información consolidada:
   • Datos del reclamo (fecha, tipo, severidad, montos)
   • Datos de la póliza (vehículo, cobertura, premium)
   • Datos del cliente (nombre, ubicación, demografía)

🔍 Muestra de datos integrados:


claim_no,first_name,last_name,vehicle_make,vehicle_model,incident_type,claim_amount,premium
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,Julian,Schwarz,TOYOTA,LAND CRUISER,Multi-vehicle Collision,5070,1330.0
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,Timo,Heinrichs,HYUNDAI,AZERA,Multi-vehicle Collision,62920,1005.0
1aec2184-3a54-4b3b-a41b-51b79feb4538,Pascal,Vogel,RANGE ROVER,VELAR 2.0 S,Parked Car,32480,4448.0
22b8e795-10e8-4981-b62c-59607a215655,Paula,Weber,FORD,RANGER,Parked Car,6030,2060.0
5ccc8c8f-d199-4cad-af49-90e45a729489,Alexander,Hofmann,KIA,SPORTAGE,Single Vehicle Collision,78100,1050.0


In [0]:
# =====================================================
# PASO C: gold.customer_claim_policy_telematics
# =====================================================
# Objetivo: Enriquecer tabla integrada con telemática del vehículo
# =====================================================

print("="*70)
print("📡 PASO C: Enriquecimiento con Telemática Agregada")
print("="*70)

# Leer tablas Gold creadas anteriormente
df_customer_claim_policy = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy")
df_aggregated_telematics = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.aggregated_telematics")

print(f"\n📊 Registros en cada tabla:")
print(f"   customer_claim_policy: {df_customer_claim_policy.count():,}")
print(f"   aggregated_telematics: {df_aggregated_telematics.count():,}")

# Unir con telemática por chassis_no
print("\n🔗 Uniendo con datos de telemática del vehículo...")
df_customer_claim_policy_telematics = df_customer_claim_policy.join(
    df_aggregated_telematics,
    df_customer_claim_policy.chassis_no == df_aggregated_telematics.chassis_no,
    "left"  # Left join para conservar todos los reclamos (algunos vehículos podrían no tener telemática)
)

# Seleccionar columnas (evitar duplicado de chassis_no)
df_customer_claim_policy_telematics = df_customer_claim_policy_telematics.select(
    # Todas las columnas de customer_claim_policy
    df_customer_claim_policy["*"],
    
    # Agregar indicadores de telemática (renombrar para claridad)
    df_aggregated_telematics.avg_speed.alias("vehicle_avg_speed"),
    df_aggregated_telematics.min_speed.alias("vehicle_min_speed"),
    df_aggregated_telematics.max_speed.alias("vehicle_max_speed"),
    df_aggregated_telematics.std_speed.alias("vehicle_std_speed"),
    df_aggregated_telematics.avg_latitude.alias("vehicle_avg_latitude"),
    df_aggregated_telematics.avg_longitude.alias("vehicle_avg_longitude"),
    df_aggregated_telematics.latitude_range.alias("vehicle_latitude_range"),
    df_aggregated_telematics.longitude_range.alias("vehicle_longitude_range"),
    df_aggregated_telematics.first_event_timestamp.alias("vehicle_first_event"),
    df_aggregated_telematics.last_event_timestamp.alias("vehicle_last_event"),
    df_aggregated_telematics.total_events.alias("vehicle_total_events"),
    df_aggregated_telematics.days_active.alias("vehicle_days_active"),
    df_aggregated_telematics.avg_events_per_day.alias("vehicle_avg_events_per_day")
)

# Agregar columna indicadora de telemática disponible
df_customer_claim_policy_telematics = df_customer_claim_policy_telematics.withColumn(
    "has_telematics",
    when(col("vehicle_avg_speed").isNotNull(), True).otherwise(False)
)

# Guardar tabla Gold final
df_customer_claim_policy_telematics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")

print(f"\n✓ Tabla {CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics creada")
print(f"  Registros totales: {df_customer_claim_policy_telematics.count():,}")
print(f"  Columnas: {len(df_customer_claim_policy_telematics.columns)}")

# Estadísticas de telemática
with_telematics = df_customer_claim_policy_telematics.filter(col("has_telematics") == True).count()
without_telematics = df_customer_claim_policy_telematics.filter(col("has_telematics") == False).count()

print(f"\n📊 Cobertura de telemática:")
print(f"   • Reclamos con telemática: {with_telematics:,} ({(with_telematics/df_customer_claim_policy_telematics.count())*100:.1f}%)")
print(f"   • Reclamos sin telemática: {without_telematics:,} ({(without_telematics/df_customer_claim_policy_telematics.count())*100:.1f}%)")

print("\n🎯 Tabla final enriquecida con:")
print("   • Información completa del reclamo")
print("   • Datos de la póliza y vehículo")
print("   • Información del cliente")
print("   • Indicadores de comportamiento del vehículo (telemática)")

# Mostrar muestra
print("\n🔍 Muestra de datos finales:")
display(df_customer_claim_policy_telematics.select(
    "claim_no", "first_name", "last_name", "vehicle_make", "vehicle_model",
    "claim_amount", "vehicle_avg_speed", "vehicle_max_speed", "has_telematics"
).limit(5))

📡 PASO C: Enriquecimiento con Telemática Agregada

📊 Registros en cada tabla:
   customer_claim_policy: 12,992
   aggregated_telematics: 11,585

🔗 Uniendo con datos de telemática del vehículo...

✓ Tabla smart_claims.gold.customer_claim_policy_telematics creada
  Registros totales: 12,992
  Columnas: 58

📊 Cobertura de telemática:
   • Reclamos con telemática: 12,992 (100.0%)
   • Reclamos sin telemática: 0 (0.0%)

🎯 Tabla final enriquecida con:
   • Información completa del reclamo
   • Datos de la póliza y vehículo
   • Información del cliente
   • Indicadores de comportamiento del vehículo (telemática)

🔍 Muestra de datos finales:


claim_no,first_name,last_name,vehicle_make,vehicle_model,claim_amount,vehicle_avg_speed,vehicle_max_speed,has_telematics
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,Julian,Schwarz,TOYOTA,LAND CRUISER,5070,39.13,71.72,true
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,Timo,Heinrichs,HYUNDAI,AZERA,62920,35.64,71.72,true
1aec2184-3a54-4b3b-a41b-51b79feb4538,Pascal,Vogel,RANGE ROVER,VELAR 2.0 S,32480,39.13,71.72,true
22b8e795-10e8-4981-b62c-59607a215655,Paula,Weber,FORD,RANGER,6030,39.13,71.72,true
5ccc8c8f-d199-4cad-af49-90e45a729489,Alexander,Hofmann,KIA,SPORTAGE,78100,32.15,57.18,true


In [0]:
# =====================================================
# VERIFICACIÓN FINAL - CAPA GOLD
# =====================================================

print("="*80)
print("✅ VERIFICACIÓN COMPLETA - CAPA GOLD")
print("="*80)
print("\n🎯 Objetivo: Tablas integradas y útiles para análisis de negocio")

# Listar todas las tablas en el esquema gold
print(f"\n📋 Tablas en {CATALOG}.{GOLD_SCHEMA}:\n")
tablas_gold = spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}").collect()

if len(tablas_gold) == 0:
    print("  ⚠️  No hay tablas en el esquema gold aún.")
    print("  👉 Ejecuta las celdas 2-4 primero para crear las tablas.")
else:
    tabla_info = []
    for tabla in tablas_gold:
        nombre_tabla = tabla['tableName']
        df_tabla = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{nombre_tabla}")
        count = df_tabla.count()
        cols = len(df_tabla.columns)
        tabla_info.append((nombre_tabla, count, cols))
    
    # Ordenar por nombre
    tabla_info.sort(key=lambda x: x[0])
    
    for nombre, count, cols in tabla_info:
        print(f"  ✓ {nombre:<40} {count:>10,} registros  |  {cols:>3} columnas")

# Verificación detallada de cada paso
print("\n" + "="*80)
print("📊 VERIFICACIÓN POR PASO")
print("="*80)

try:
    # PASO A
    print("\n🚗 PASO A: aggregated_telematics")
    df_agg_tel = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.aggregated_telematics")
    print(f"   ✓ Vehículos agregados: {df_agg_tel.count():,}")
    print(f"   ✓ Indicadores por vehículo: {len(df_agg_tel.columns)} columnas")
    print("   • Velocidad: avg, min, max, std")
    print("   • Ubicación: coordenadas promedio y dispersión")
    print("   • Actividad: eventos totales, días activos")
except Exception as e:
    print(f"   ❌ Error al leer aggregated_telematics: {str(e)}")

try:
    # PASO B
    print("\n📄 PASO B: customer_claim_policy")
    df_ccp = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy")
    print(f"   ✓ Reclamos integrados: {df_ccp.count():,}")
    print(f"   ✓ Columnas totales: {len(df_ccp.columns)}")
    print("   • Consolidación: claims + policies + customers")
    print("   • Información completa de reclamo, póliza y cliente")
except Exception as e:
    print(f"   ❌ Error al leer customer_claim_policy: {str(e)}")

try:
    # PASO C
    print("\n📡 PASO C: customer_claim_policy_telematics")
    df_ccpt = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")
    print(f"   ✓ Registros finales: {df_ccpt.count():,}")
    print(f"   ✓ Columnas totales: {len(df_ccpt.columns)}")
    
    with_tel = df_ccpt.filter(col("has_telematics") == True).count()
    without_tel = df_ccpt.filter(col("has_telematics") == False).count()
    print(f"   • Reclamos con telemática: {with_tel:,} ({(with_tel/df_ccpt.count())*100:.1f}%)")
    print(f"   • Reclamos sin telemática: {without_tel:,} ({(without_tel/df_ccpt.count())*100:.1f}%)")
except Exception as e:
    print(f"   ❌ Error al leer customer_claim_policy_telematics: {str(e)}")

# Resumen final
print("\n" + "="*80)
print("🎉 CAPA GOLD COMPLETADA")
print("="*80)

tablas_esperadas = ["aggregated_telematics", "customer_claim_policy", "customer_claim_policy_telematics"]
tablas_existentes = [t['tableName'] for t in tablas_gold]
tablas_creadas = [t for t in tablas_esperadas if t in tablas_existentes]

print(f"\nTablas creadas: {len(tablas_creadas)}/{len(tablas_esperadas)}\n")
for tabla in tablas_esperadas:
    status = "✅" if tabla in tablas_existentes else "❌"
    print(f"   {status} gold.{tabla}")

if len(tablas_creadas) == len(tablas_esperadas):
    print("\n✅ Todas las tablas Gold creadas exitosamente")
    print("\n💡 Casos de uso:")
    print("   • Análisis de comportamiento de vehículos")
    print("   • Predicción de reclamos fraudulentos")
    print("   • Segmentación de clientes y pólizas")
    print("   • Dashboards de negocio y reportes ejecutivos")
    print("   • Modelos de Machine Learning")
else:
    print(f"\n⚠️  Faltan {len(tablas_esperadas) - len(tablas_creadas)} tabla(s) por crear")
    print("   Ejecuta las celdas anteriores para completar la capa Gold")

✅ VERIFICACIÓN COMPLETA - CAPA GOLD

🎯 Objetivo: Tablas integradas y útiles para análisis de negocio

📋 Tablas en smart_claims.gold:

  ✓ aggregated_telematics                        11,585 registros  |   14 columnas
  ✓ customer_claim_policy                        12,992 registros  |   44 columnas
  ✓ customer_claim_policy_telematics             12,992 registros  |   58 columnas

📊 VERIFICACIÓN POR PASO

🚗 PASO A: aggregated_telematics
   ✓ Vehículos agregados: 11,585
   ✓ Indicadores por vehículo: 14 columnas
   • Velocidad: avg, min, max, std
   • Ubicación: coordenadas promedio y dispersión
   • Actividad: eventos totales, días activos

📄 PASO B: customer_claim_policy
   ✓ Reclamos integrados: 12,992
   ✓ Columnas totales: 44
   • Consolidación: claims + policies + customers
   • Información completa de reclamo, póliza y cliente

📡 PASO C: customer_claim_policy_telematics
   ✓ Registros finales: 12,992
   ✓ Columnas totales: 58
   • Reclamos con telemática: 12,992 (100.0%)
   • Rec

In [0]:
# =====================================================
# VERIFICACIÓN EXHAUSTIVA - REQUISITOS DE NEGOCIO
# =====================================================

print("="*80)
print("🔍 VERIFICACIÓN EXHAUSTIVA - REQUISITOS DE NEGOCIO")
print("="*80)

# =====================================================
# 1. VERIFICAR QUE LA TABLA FINAL DE GOLD EXISTE
# =====================================================
print("\n1️⃣  VERIFICAR QUE LA TABLA FINAL DE GOLD EXISTE")
print("="*80)

try:
    df_gold = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")
    print(f"✅ Tabla gold.customer_claim_policy_telematics EXISTE")
    print(f"   📊 Registros: {df_gold.count():,}")
    print(f"   📋 Columnas: {len(df_gold.columns)}")
    tabla_existe = True
except Exception as e:
    print(f"❌ Error: La tabla NO existe - {str(e)}")
    tabla_existe = False

# =====================================================
# 2. VERIFICAR VINCULACIÓN RECLAMO-PÓLIZA
# =====================================================
if tabla_existe:
    print("\n2️⃣  VERIFICAR VINCULACIÓN RECLAMO-PÓLIZA")
    print("="*80)
    
    # Contar reclamos con póliza
    claims_con_poliza = df_gold.filter(col("policy_no").isNotNull()).count()
    claims_sin_poliza = df_gold.filter(col("policy_no").isNull()).count()
    
    print(f"   ✓ Reclamos con póliza vinculada: {claims_con_poliza:,}")
    print(f"   ✓ Reclamos sin póliza: {claims_sin_poliza:,}")
    
    if claims_sin_poliza == 0:
        print(f"   ✅ PERFECTO: Todos los reclamos tienen póliza vinculada")
    else:
        print(f"   ⚠️  Algunos reclamos no tienen póliza ({(claims_sin_poliza/df_gold.count())*100:.1f}%)")
    
    # Verificar campos clave de póliza
    print("\n   Campos clave de póliza disponibles:")
    campos_poliza = ['policy_no', 'policy_type', 'vehicle_make', 'vehicle_model', 
                      'chassis_no', 'premium', 'sum_insured']
    for campo in campos_poliza:
        no_nulos = df_gold.filter(col(campo).isNotNull()).count()
        print(f"      • {campo}: {no_nulos:,} registros ({(no_nulos/df_gold.count())*100:.1f}%)")

# =====================================================
# 3. VERIFICAR VINCULACIÓN RECLAMO-CLIENTE
# =====================================================
if tabla_existe:
    print("\n3️⃣  VERIFICAR VINCULACIÓN RECLAMO-CLIENTE")
    print("="*80)
    
    # Contar reclamos con cliente
    claims_con_cliente = df_gold.filter(col("customer_id").isNotNull()).count()
    claims_sin_cliente = df_gold.filter(col("customer_id").isNull()).count()
    
    print(f"   ✓ Reclamos con cliente vinculado: {claims_con_cliente:,}")
    print(f"   ✓ Reclamos sin cliente: {claims_sin_cliente:,}")
    
    if claims_sin_cliente == 0:
        print(f"   ✅ PERFECTO: Todos los reclamos tienen cliente vinculado")
    else:
        print(f"   ⚠️  Algunos reclamos no tienen cliente ({(claims_sin_cliente/df_gold.count())*100:.1f}%)")
    
    # Verificar campos clave de cliente
    print("\n   Campos clave de cliente disponibles:")
    campos_cliente = ['customer_id', 'first_name', 'last_name', 'date_of_birth', 
                       'borough', 'address']
    for campo in campos_cliente:
        no_nulos = df_gold.filter(col(campo).isNotNull()).count()
        print(f"      • {campo}: {no_nulos:,} registros ({(no_nulos/df_gold.count())*100:.1f}%)")

# =====================================================
# 4. VERIFICAR UNIÓN CON TELEMÁTICA
# =====================================================
if tabla_existe:
    print("\n4️⃣  VERIFICAR UNIÓN CON TELEMÁTICA POR VEHÍCULO")
    print("="*80)
    
    # Contar reclamos con telemática
    claims_con_telematica = df_gold.filter(col("has_telematics") == True).count()
    claims_sin_telematica = df_gold.filter(col("has_telematics") == False).count()
    
    print(f"   ✓ Reclamos con telemática: {claims_con_telematica:,} ({(claims_con_telematica/df_gold.count())*100:.1f}%)")
    print(f"   ✓ Reclamos sin telemática: {claims_sin_telematica:,} ({(claims_sin_telematica/df_gold.count())*100:.1f}%)")
    
    if claims_con_telematica > 0:
        print(f"   ✅ CORRECTO: La telemática se unió exitosamente por chassis_no")
    
    # Verificar indicadores de telemática
    print("\n   Indicadores de telemática disponibles:")
    campos_telematica = ['vehicle_avg_speed', 'vehicle_max_speed', 'vehicle_min_speed',
                          'vehicle_avg_latitude', 'vehicle_avg_longitude', 
                          'vehicle_total_events', 'vehicle_days_active']
    for campo in campos_telematica:
        no_nulos = df_gold.filter(col(campo).isNotNull()).count()
        print(f"      • {campo}: {no_nulos:,} registros")

# =====================================================
# 5. VERIFICAR SENTIDO PARA ANÁLISIS DE NEGOCIO
# =====================================================
if tabla_existe:
    print("\n5️⃣  VERIFICAR SENTIDO PARA ANÁLISIS DE NEGOCIO")
    print("="*80)
    
    # Verificar que la tabla tenga dimensiones relevantes para análisis
    dimensiones = {
        'Reclamo': ['claim_no', 'claim_date', 'incident_date', 'incident_type', 'claim_amount'],
        'Cliente': ['customer_id', 'first_name', 'last_name', 'borough', 'age'],
        'Póliza': ['policy_no', 'policy_type', 'vehicle_make', 'premium'],
        'Vehículo': ['chassis_no', 'vehicle_make', 'vehicle_model'],
        'Telemática': ['vehicle_avg_speed', 'vehicle_max_speed', 'vehicle_total_events'],
        'Fraude': ['fraud_reported', 'suspicious indicators']
    }
    
    print("\n   Dimensiones disponibles para análisis:")
    cols_disponibles = df_gold.columns
    for dimension, campos in dimensiones.items():
        campos_existentes = [c for c in campos if c in cols_disponibles or any(c in col for col in cols_disponibles)]
        print(f"      ✅ {dimension}: {len(campos_existentes)} campos disponibles")
    
    print("\n   ✅ La tabla tiene estructura completa para análisis multidimensional")

print("\n" + "="*80)
print("📊 RESUMEN DE VERIFICACIÓN")
print("="*80)

if tabla_existe:
    print("\n✅ TODOS LOS REQUISITOS CUMPLIDOS:")
    print("   ✓ Tabla final de Gold existe")
    print("   ✓ Reclamos vinculados con pólizas")
    print("   ✓ Reclamos vinculados con clientes")
    print("   ✓ Telemática unida por vehículo (chassis_no)")
    print("   ✓ Tabla lista para análisis de negocio")
    print("\n🎯 Base principal del caso LISTA para la siguiente fase")
else:
    print("\n❌ Falta crear la tabla final de Gold")
    print("   Ejecuta las celdas 2-4 para crear las tablas")

🔍 VERIFICACIÓN EXHAUSTIVA - REQUISITOS DE NEGOCIO

1️⃣  VERIFICAR QUE LA TABLA FINAL DE GOLD EXISTE
✅ Tabla gold.customer_claim_policy_telematics EXISTE
   📊 Registros: 12,992
   📋 Columnas: 58

2️⃣  VERIFICAR VINCULACIÓN RECLAMO-PÓLIZA
   ✓ Reclamos con póliza vinculada: 12,992
   ✓ Reclamos sin póliza: 0
   ✅ PERFECTO: Todos los reclamos tienen póliza vinculada

   Campos clave de póliza disponibles:
      • policy_no: 12,992 registros (100.0%)
      • policy_type: 12,992 registros (100.0%)
      • vehicle_make: 12,992 registros (100.0%)
      • vehicle_model: 12,992 registros (100.0%)
      • chassis_no: 12,992 registros (100.0%)
      • premium: 12,992 registros (100.0%)
      • sum_insured: 12,992 registros (100.0%)

3️⃣  VERIFICAR VINCULACIÓN RECLAMO-CLIENTE
   ✓ Reclamos con cliente vinculado: 12,992
   ✓ Reclamos sin cliente: 0
   ✅ PERFECTO: Todos los reclamos tienen cliente vinculado

   Campos clave de cliente disponibles:
      • customer_id: 12,992 registros (100.0%)
     

In [0]:
# =====================================================
# DEMOSTRACIÓN: PREGUNTAS DE NEGOCIO
# =====================================================

print("="*80)
print("💬 DEMOSTRACIÓN: LA TABLA GOLD RESPONDE PREGUNTAS DE NEGOCIO")
print("="*80)

df_gold = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.customer_claim_policy_telematics")

# =====================================================
# PREGUNTA 1: ¿Qué cliente está asociado a cada reclamo?
# =====================================================
print("\n" + "="*80)
print("👤 PREGUNTA 1: ¿Qué cliente está asociado a cada reclamo?")
print("="*80)

print("\nRespuesta: Tomemos 5 reclamos de ejemplo...\n")

df_pregunta1 = df_gold.select(
    "claim_no",
    "customer_id",
    concat(col("first_name"), lit(" "), col("last_name")).alias("nombre_completo"),
    "date_of_birth",
    "borough",
    "address"
).limit(5)

print("Reclamo → Cliente (con información completa):")
display(df_pregunta1)

print("\n✅ Cada reclamo tiene toda la información del cliente:")
print("   • Nombre completo")
print("   • Fecha de nacimiento")
print("   • Ubicación (borough, address)")

# =====================================================
# PREGUNTA 2: ¿Qué póliza cubre ese reclamo?
# =====================================================
print("\n" + "="*80)
print("📄 PREGUNTA 2: ¿Qué póliza cubre ese reclamo?")
print("="*80)

print("\nRespuesta: Tomemos 5 reclamos de ejemplo...\n")

df_pregunta2 = df_gold.select(
    "claim_no",
    "policy_no",
    "policy_type",
    concat(col("vehicle_make"), lit(" "), col("vehicle_model")).alias("vehiculo"),
    "chassis_no",
    "premium",
    "sum_insured",
    "deductible"
).limit(5)

print("Reclamo → Póliza (con información completa):")
display(df_pregunta2)

print("\n✅ Cada reclamo tiene toda la información de la póliza:")
print("   • Número de póliza y tipo")
print("   • Vehículo asegurado (marca, modelo, chassis)")
print("   • Cobertura financiera (premium, suma asegurada, deducible)")

# =====================================================
# PREGUNTA 3: ¿Cuál fue el comportamiento telemático del vehículo?
# =====================================================
print("\n" + "="*80)
print("📡 PREGUNTA 3: ¿Cuál fue el comportamiento telemático del vehículo?")
print("="*80)

print("\nRespuesta: Tomemos 5 reclamos con telemática...\n")

df_pregunta3 = df_gold.filter(col("has_telematics") == True).select(
    "claim_no",
    concat(col("vehicle_make"), lit(" "), col("vehicle_model")).alias("vehiculo"),
    "chassis_no",
    "vehicle_avg_speed",
    "vehicle_max_speed",
    "vehicle_total_events",
    "vehicle_days_active",
    "vehicle_avg_events_per_day"
).limit(5)

print("Reclamo → Comportamiento Telemático del Vehículo:")
display(df_pregunta3)

print("\n✅ Cada reclamo tiene indicadores de comportamiento del vehículo:")
print("   • Velocidad promedio y máxima")
print("   • Número total de eventos registrados")
print("   • Días de actividad")
print("   • Frecuencia de eventos por día")

# =====================================================
# EJEMPLO INTEGRADO: Vista 360° de un reclamo
# =====================================================
print("\n" + "="*80)
print("🎯 EJEMPLO: VISTA 360° DE UN RECLAMO")
print("="*80)

print("\nVista completa de UN reclamo con TODA la información integrada:\n")

# Tomar un reclamo de ejemplo
reclamo_ejemplo = df_gold.limit(1).collect()[0]

print(f"📋 RECLAMO: {reclamo_ejemplo['claim_no']}")
print("\n👤 CLIENTE:")
print(f"   • Nombre: {reclamo_ejemplo['first_name']} {reclamo_ejemplo['last_name']}")
print(f"   • ID: {reclamo_ejemplo['customer_id']}")
print(f"   • Ubicación: {reclamo_ejemplo['borough']}, {reclamo_ejemplo['neighborhood']}")

print("\n📄 PÓLIZA:")
print(f"   • Número: {reclamo_ejemplo['policy_no']}")
print(f"   • Tipo: {reclamo_ejemplo['policy_type']}")
print(f"   • Vehículo: {reclamo_ejemplo['vehicle_make']} {reclamo_ejemplo['vehicle_model']} ({reclamo_ejemplo['vehicle_year']})")
print(f"   • Premium: ${reclamo_ejemplo['premium']:,.2f}")
print(f"   • Suma asegurada: ${reclamo_ejemplo['sum_insured']:,.2f}")

print("\n🚨 RECLAMO:")
print(f"   • Fecha: {reclamo_ejemplo['claim_date']}")
print(f"   • Tipo de incidente: {reclamo_ejemplo['incident_type']}")
print(f"   • Severidad: {reclamo_ejemplo['incident_severity']}")
print(f"   • Monto: ${reclamo_ejemplo['claim_amount']:,}")
print(f"   • Fraude reportado: {reclamo_ejemplo['fraud_reported']}")

if reclamo_ejemplo['has_telematics']:
    print("\n📡 TELEMÁTICA DEL VEHÍCULO:")
    print(f"   • Velocidad promedio: {reclamo_ejemplo['vehicle_avg_speed']:.2f} km/h")
    print(f"   • Velocidad máxima: {reclamo_ejemplo['vehicle_max_speed']:.2f} km/h")
    print(f"   • Eventos registrados: {reclamo_ejemplo['vehicle_total_events']:,}")
    print(f"   • Días activos: {reclamo_ejemplo['vehicle_days_active']}")
else:
    print("\n⚠️  TELEMÁTICA: No disponible para este vehículo")

print("\n" + "="*80)
print("✅ CONCLUSIÓN")
print("="*80)
print("\nLa tabla gold.customer_claim_policy_telematics permite:")
print("\n1️⃣  Identificar el cliente de cada reclamo con información completa")
print("2️⃣  Conocer la póliza que cubre el reclamo con todos los detalles")
print("3️⃣  Analizar el comportamiento telemático del vehículo involucrado")
print("4️⃣  Realizar análisis 360° para detección de fraudes")
print("5️⃣  Generar dashboards ejecutivos y reportes")
print("6️⃣  Entrenar modelos de Machine Learning")
print("\n🎉 LA BASE PRINCIPAL DEL CASO ESTÁ LISTA PARA LA SIGUIENTE FASE")

💬 DEMOSTRACIÓN: LA TABLA GOLD RESPONDE PREGUNTAS DE NEGOCIO

👤 PREGUNTA 1: ¿Qué cliente está asociado a cada reclamo?

Respuesta: Tomemos 5 reclamos de ejemplo...

Reclamo → Cliente (con información completa):


claim_no,customer_id,nombre_completo,date_of_birth,borough,address
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,253.0,Julian Schwarz,2005-01-11,MANHATTAN,"UPPER WEST SIDE, MANHATTAN, 10025"
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,7957.0,Timo Heinrichs,1971-04-03,BRONX,"KINGSBRIDGE AND RIVERDALE, BRONX, 10463"
1aec2184-3a54-4b3b-a41b-51b79feb4538,4250.0,Pascal Vogel,2000-01-01,STATEN ISLAND,"SOUTH SHORE, STATEN ISLAND, 10307"
22b8e795-10e8-4981-b62c-59607a215655,3304.0,Paula Weber,2012-03-08,BROOKLYN,"BOROUGH PARK, BROOKLYN, 11230"
5ccc8c8f-d199-4cad-af49-90e45a729489,8647.0,Alexander Hofmann,2010-11-12,BROOKLYN,"BOROUGH PARK, BROOKLYN, 11218"



✅ Cada reclamo tiene toda la información del cliente:
   • Nombre completo
   • Fecha de nacimiento
   • Ubicación (borough, address)

📄 PREGUNTA 2: ¿Qué póliza cubre ese reclamo?

Respuesta: Tomemos 5 reclamos de ejemplo...

Reclamo → Póliza (con información completa):


claim_no,policy_no,policy_type,vehiculo,chassis_no,premium,sum_insured,deductible
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,102049729,COMP,TOYOTA LAND CRUISER,JTEHJ09J825052652,1330.0,10000.0,2000
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,102147750,COMP,HYUNDAI AZERA,KMHFH41B9EA307612,1005.0,25600.0,500
1aec2184-3a54-4b3b-a41b-51b79feb4538,102139065,COMP,RANGE ROVER VELAR 2.0 S,SALYA2BX1JA733524,4448.0,205360.0,2000
22b8e795-10e8-4981-b62c-59607a215655,102099757,COMP,FORD RANGER,AFAFP1LPXDJU88187,2060.0,42000.0,500
5ccc8c8f-d199-4cad-af49-90e45a729489,102071474,COMP,KIA SPORTAGE,KNAPB8110B7114716,1050.0,32000.0,2000



✅ Cada reclamo tiene toda la información de la póliza:
   • Número de póliza y tipo
   • Vehículo asegurado (marca, modelo, chassis)
   • Cobertura financiera (premium, suma asegurada, deducible)

📡 PREGUNTA 3: ¿Cuál fue el comportamiento telemático del vehículo?

Respuesta: Tomemos 5 reclamos con telemática...

Reclamo → Comportamiento Telemático del Vehículo:


claim_no,vehiculo,chassis_no,vehicle_avg_speed,vehicle_max_speed,vehicle_total_events,vehicle_days_active,vehicle_avg_events_per_day
f59f665f-c82f-4aa3-98b5-060c7a8f6d19,TOYOTA LAND CRUISER,JTEHJ09J825052652,39.13,71.72,60,0,60.0
f051f2ae-c92e-44f9-9bb8-3ab60b19341b,HYUNDAI AZERA,KMHFH41B9EA307612,35.64,71.72,120,274,0.44
1aec2184-3a54-4b3b-a41b-51b79feb4538,RANGE ROVER VELAR 2.0 S,SALYA2BX1JA733524,39.13,71.72,60,0,60.0
22b8e795-10e8-4981-b62c-59607a215655,FORD RANGER,AFAFP1LPXDJU88187,39.13,71.72,60,0,60.0
5ccc8c8f-d199-4cad-af49-90e45a729489,KIA SPORTAGE,KNAPB8110B7114716,32.15,57.18,60,0,60.0



✅ Cada reclamo tiene indicadores de comportamiento del vehículo:
   • Velocidad promedio y máxima
   • Número total de eventos registrados
   • Días de actividad
   • Frecuencia de eventos por día

🎯 EJEMPLO: VISTA 360° DE UN RECLAMO

Vista completa de UN reclamo con TODA la información integrada:

📋 RECLAMO: f59f665f-c82f-4aa3-98b5-060c7a8f6d19

👤 CLIENTE:
   • Nombre: Julian Schwarz
   • ID: 253.0
   • Ubicación: MANHATTAN, UPPER WEST SIDE

📄 PÓLIZA:
   • Número: 102049729
   • Tipo: COMP
   • Vehículo: TOYOTA LAND CRUISER (2002.0)
   • Premium: $1,330.00
   • Suma asegurada: $10,000.00

🚨 RECLAMO:
   • Fecha: 2016-01-26
   • Tipo de incidente: Multi-vehicle Collision
   • Severidad: Minor Damage
   • Monto: $5,070
   • Fraude reportado: False

📡 TELEMÁTICA DEL VEHÍCULO:
   • Velocidad promedio: 39.13 km/h
   • Velocidad máxima: 71.72 km/h
   • Eventos registrados: 60
   • Días activos: 0

✅ CONCLUSIÓN

La tabla gold.customer_claim_policy_telematics permite:

1️⃣  Identificar el cli